In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [4]:
import numpy as np
import pandas as pd
import os
import mlflow
import mlflow.sklearn
import dagshub
from catboost import CatBoostClassifier
print("Imports OK ✓")

Imports OK ✓


In [5]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "kende23"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "be7a74e194c6b7c0d666a4938cfee49142c7d603"
mlflow.set_tracking_uri("https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow")
dagshub.init(repo_owner="kende23", repo_name="ieee-cis-fraud-detection", mlflow=True)
model_cat = mlflow.sklearn.load_model("models:/CatBoost_Fraud_Pipeline/1")
print(f"Model type: {type(model_cat)}")print("Connected to DagsHub ✓")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d79e3abc-f3b7-461f-b9a7-af55d16e1754&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=a218f8e3a0e08221ba73809e9b4401e5fba8e1a1b7ec947a69e8a4e849eb769f




Accessing as kende23

Initialized MLflow to track repo "kende23/ieee-cis-fraud-detection"

Repository kende23/ieee-cis-fraud-detection initialized!

Connected to DagsHub ✓


In [13]:
model_cat = mlflow.sklearn.load_model("models:/CatBoost_Fraud_Pipeline/1")
print(f"Model type: {type(model_cat)}")

print("Cat feature indices:", model_cat.get_cat_feature_indices())
print("Feature names:", model_cat.feature_names_)
print("Total features:", len(model_cat.feature_names_))


Model type: <class 'catboost.core.CatBoostClassifier'>
Cat feature indices: [20, 22, 23, 26, 40, 42, 43, 46, 47, 50]
Feature names: ['C13', 'card1_freq', 'C14', 'D1n', 'card2', 'card1', 'addr1', 'C1', 'C6', 'dist1', 'C2', 'D1', 'C11', 'D15n', 'card5', 'D2n', 'D11', 'TransactionAmt', 'TransactionAmt_log', 'V285', 'M5', 'D10n', 'card6', 'M6', 'C5', 'D2', 'M4', 'V91', 'TransactionDay', 'C9', 'D15', 'D10', 'V90', 'V310', 'V83', 'D4', 'V82', 'card3', 'D13', 'C3', 'M3', 'V29', 'R_email_provider', 'card4', 'D8', 'cents', 'P_email_provider', 'P_emaildomain', 'V49', 'V317', 'ProductCD']
Total features: 51


In [15]:
from catboost import Pool

print("Predicting...")

# build pool with explicit cat feature indices
cat_indices = model_cat.get_cat_feature_indices()
cat_names   = [selected_features[i] for i in cat_indices]
print(f"Cat indices: {cat_indices}")
print(f"Cat names:   {cat_names}")

# only fill actual string cols, everything else leave as NaN
test_data = test[selected_features].copy()

for col in cat_names:
    test_data[col] = test_data[col].fillna("missing").astype(str)

# all other cols — make sure no string "missing" values
non_cat_cols = [c for c in selected_features if c not in cat_names]
for col in non_cat_cols:
    # if accidentally converted to string, convert back
    if test_data[col].dtype == object:
        test_data[col] = pd.to_numeric(test_data[col], errors="coerce")

print(f"\nDtypes of non-cat cols that are object:")
obj_cols = [c for c in non_cat_cols if test_data[c].dtype == object]
print(f"  {obj_cols}")

pool = Pool(data=test_data, cat_features=cat_indices)
test_probs = model_cat.predict_proba(pool)[:, 1]

print(f"  Predictions shape: {test_probs.shape}")
print(f"  Fraud rate: {test_probs.mean():.4f}")
print(f"  Min: {test_probs.min():.4f}, Max: {test_probs.max():.4f}")

submission = pd.DataFrame({
    "TransactionID": test_tr["TransactionID"].values,
    "isFraud": test_probs
})

print(f"\nSubmission shape: {submission.shape}")
print(submission.head(10))

submission.to_csv("submission_catboost_inference.csv", index=False)
print("\nsubmission_catboost_inference.csv saved ✓")
print("Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions")

Predicting...
Cat indices: [20, 22, 23, 26, 40, 42, 43, 46, 47, 50]
Cat names:   ['M5', 'card6', 'M6', 'M4', 'M3', 'R_email_provider', 'card4', 'P_email_provider', 'P_emaildomain', 'ProductCD']

Dtypes of non-cat cols that are object:
  []
  Predictions shape: (506691,)
  Fraud rate: 0.2329
  Min: 0.0002, Max: 0.9998

Submission shape: (506691, 2)
   TransactionID   isFraud
0        3663549  0.039662
1        3663550  0.193051
2        3663551  0.341784
3        3663552  0.103810
4        3663553  0.049368
5        3663554  0.192997
6        3663555  0.203588
7        3663556  0.310888
8        3663557  0.032035
9        3663558  0.172482

submission_catboost_inference.csv saved ✓
Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions
